In [ ]:
import hydra
from omegaconf import OmegaConf
import numpy as np
import torch
from matplotlib import pyplot as plt
from segmentation.environment import SegmentationEnv
from segmentation.learners import DQNLearner

# Load configuration

In [ ]:
hydra.initialize(config_path="conf", version_base=None)
cfg = hydra.compose(config_name="config")

print(OmegaConf.to_yaml(cfg))

In [ ]:
data = np.load(cfg.dataset)
samples = data["sequence"]
gt_break_points = data["labels"]

plt.figure(figsize=(12, 6))
plt.plot(samples)
for bkp in gt_break_points:
    plt.axvline(bkp, color='k')

# Create Environent and Agent 

In [ ]:
env = SegmentationEnv(
            samples=samples, 
            gt_break_points=gt_break_points,
            level_wavelet=cfg.level_wavelet,
            window_size=cfg.window_size,
            tolerance=cfg.tolerance,
            args=cfg) 

state, info = env.reset()
learner = DQNLearner(num_samples=samples.shape[0], gt_break_points=gt_break_points, args=cfg, options={"mode": 'eval'})


learner.model.load_state_dict(torch.load(cfg.checkpoint_root))

break_points = env.segmenter.get_break_points()

plt.figure(figsize=(12, 6))
plt.plot(samples)
for bkp in break_points:
    plt.axvline(bkp, color='r', linestyle='--')
for bkp in gt_break_points:
    plt.axvline(bkp, color='k')

print(f"Initial Precision: {info['precision']:.4f}, Recall: {info['recall']:.4f}, F1: {info['f1']:.4f}")

In [ ]:
total_reward = 0
done = False
step = 0
while not done:
    action = learner.get_candidate(state)
    next_state, reward, done, terminated, info = env.step(action)
    total_reward += reward
    state = next_state
    step += 1
    print(f"Step Reward {step}: {reward:.4f}, Precision: {info['precision']:.4f}, Recall: {info['recall']:.4f}, F1: {info['f1']:.4f}")
    
print(f"Total Reward: {total_reward:.4f}")


In [ ]:
break_points = env.segmenter.get_break_points()

plt.figure(figsize=(12, 6))
plt.plot(samples)
for bkp in gt_break_points:
    plt.axvline(bkp, color='k')
for bkp in break_points:
    plt.axvline(bkp, color='r', linestyle='--')